In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import numpy as np
from topoflow.flow.paths import GaussianConditionalProbabilityPath
from topoflow.flow.schedulers import LinearAlpha, LinearBeta
from topoflow.utils.visualization import visualize_data_rdkit, visualize_data, visualize_trajectory, visualize_data_matplotlib
from topoflow.flow.flow_models import FlowModel

import argparse
import os
import numpy as np
import torch
import lightning.pytorch as pl
from topoflow.data.dataloader import DataLoaderSampleable
from topoflow.data.datasets import CrossDocked
from topoflow.data import MockDatamodule
from topoflow.flow.schedulers import LinearAlpha, LinearBeta
from topoflow.flow.paths import GaussianConditionalProbabilityPath
from topoflow.flow.vector_field_models import TransformerVectorField
from topoflow.flow.flow_models import FlowModel
import topoflow.utils.constants as constants
from topoflow.flow.utils import InverseTimeFactor
from topoflow.data.augmentations import RandomRotateTranslate
from topoflow.utils.metrics import get_distributions


In [3]:
data_name = "crossdock_full"
augment = False
batch_size = 1
data_dir = "../data/CrossDocked/"

if data_name == "crossdock_full":
    train_dataset = CrossDocked(
        root=data_dir,
        split="train",
        transform=RandomRotateTranslate() if augment else None,
    )
    val_dataset = CrossDocked(root=data_dir, split="val")

else:
    raise ValueError(f"Unsupported dataset: {data_name}")
datamodule = MockDatamodule(
    train_length=len(train_dataset),
    val_length=0, #len(val_dataset), #TODO: fix this
    batch_size=batch_size,
    test_length=100,
)

alpha = LinearAlpha()
beta = LinearBeta()
p_data = DataLoaderSampleable(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=8,
)

distribution_hist = np.load("../data/CrossDocked/processed/size_distribution.npy", allow_pickle=True)

path = GaussianConditionalProbabilityPath(p_data=p_data, alpha=alpha, beta=beta, target_dim=14, distribution_hist=distribution_hist)
checkpoint_path = "../checkpoints/topoflow/topoflow-epoch=2326-v2.ckpt"

#Load the model from the checkpoint
model = FlowModel.load_from_checkpoint(checkpoint_path, path=path, norm_path="../data/CrossDocked/processed/normalization_params.npz")

Loaded dataset from ../data/CrossDocked/processed/train_crossdocked_pyg.pt.
Loaded dataset from ../data/CrossDocked/processed/val_crossdocked_pyg.pt.


/home/marco/miniforge3/envs/topoflow/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Entropy of n_nodes: H[N] 9.109623908996582


/home/marco/miniforge3/envs/topoflow/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'vector_field_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['vector_field_model'])`.
/home/marco/miniforge3/envs/topoflow/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'time_factor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['time_factor'])`.


In [11]:
mock_train = datamodule.train_dataloader()

for batch in mock_train:
    batch_size = batch.shape[0]

    z, batch_lig, batch_pocket, batch_size = model.path.sample_conditioning_variable(
        batch_size
    )
    t, t_batched = model.path.sample_time(batch_size, batch_lig, batch_pocket)
    print(f"t: {t}")
    x = model.path.sample_conditional_path(z, t_batched)
    u_t_ref = model.path.conditional_vector_field(x, z, t_batched)
    # u_t_theta = model.vector_field_model(x, t, batch_lig, batch_pocket)
    
    device = x.device

    batch_x = torch.cat([batch_lig, batch_pocket], dim=0)

    # Build type labels: 1 for ligand, 0 for pocket
    type_labels = torch.cat(
        [
            torch.ones(len(batch_lig), dtype=torch.long),
            torch.zeros(len(batch_pocket), dtype=torch.long),
        ]
    ).to(device)

    # Get learned type embedding
    type_embed = model.vector_field_model.type_embed(type_labels)  # [num_x, type_embed_dim]

    x_embed = model.vector_field_model.x_embed(x)  # [num_x, h_dim]
    print(f"x_embed: {x_embed}")
    # coords_embed = self.coords_embed(x[:, :self.spatial_dim])  # [num_x, h_dim]
    # atom_embed = self.atom_embed(x[:, self.spatial_dim:])  # [num_x, h_dim]
    
    time_embed = model.vector_field_model.time_embed(t)  # [batch_size, h_dim]
    print(f"time_embed: {time_embed}")
    # Now assign each node the right t embedding via batch_x
    t_per_x = time_embed[batch_x]  # [num_x, h_dim]
    
    # Sum up embeddings
    x = x_embed + type_embed + t_per_x  # [num_x, h_dim]
    
    # Create attention mask: prevent attention across graphs
    attn_mask = (batch_x[:, None] != batch_x[None, :]).float().to(device)
    attn_mask = attn_mask.masked_fill(attn_mask > 0, float("-inf"))

    # Transformer forward
    x_encoded = model.vector_field_model.transformer(x.unsqueeze(0), mask=attn_mask).squeeze(0)
    print(f"x_encoded: {x_encoded}")
    # Output projections
    coords_out = model.vector_field_model.out_coord_linear(x_encoded)  # [num_x, spatial_dim]
    atom_type_out = model.vector_field_model.out_atom_type_linear(x_encoded)  # [num_x, atom_dim]

    res = torch.cat([coords_out, atom_type_out], dim=-1)
    print(f"res: {res}")


t: tensor([[0.7065]], device='cuda:0')
x_embed: tensor([[-0.0551,  0.6875,  1.4160,  ...,  2.5386,  0.0356, -1.6507],
        [ 2.0854,  0.1738,  1.6293,  ...,  2.3652, -3.7054, -2.4658],
        [ 4.1241,  0.6683, -2.0083,  ...,  1.4347, -1.8783, -1.8629],
        ...,
        [ 5.2796,  3.1911, -0.4801,  ...,  0.6684, -2.0090, -2.1922],
        [-0.2967,  2.8127,  2.8430,  ...,  1.8107, -0.4084,  2.1697],
        [-0.5347,  0.1272, -0.0126,  ..., -3.8425,  1.0769, -1.1184]],
       device='cuda:0', grad_fn=<AddmmBackward0>)
time_embed: tensor([[ 0.0763, -0.2797, -0.1442,  0.2065,  0.8047, -0.4135,  0.3953, -0.9555,
         -0.9919, -0.8777,  0.4303,  0.4186, -0.6268,  0.3809,  0.3377, -0.9936,
          0.2494,  0.9925,  0.2317, -0.7104, -0.9966, -0.6897, -0.1474,  0.3605,
          0.7204,  0.9214,  0.9960,  0.9854,  0.9245,  0.8384,  0.7434,  0.6492,
         -0.9971,  0.9601,  0.9895, -0.9784, -0.5937, -0.9105,  0.9185,  0.2951,
         -0.1271,  0.4793,  0.9027, -0.9082,  0.779